# Hands-on — Aula 04: Agente ReAct sem framework

Dois hands-on no mesmo notebook (o modelo carrega uma vez e serve os dois):

| Seção | Tema | Slide |
|---|---|---|
| 1 | Agente ReAct mínimo: o loop pensar→agir→observar | 10 |
| 2 | Contratos: schema pydantic, validação, limites e auditoria | 13–14, 19 |

**Modelo:** `Qwen2.5-1.5B-Instruct` · **Ferramentas** (todas somente leitura,
definidas neste notebook): `consultar_clima` (dicionário local) ·
`buscar_documento` (busca lexical nos docs da Aula 02) · `calculadora`
(AST com whitelist — **nunca** `eval`) · `consultar_cotacao` (propositalmente quebrada).

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import torch
from pydantic import BaseModel, Field, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # silencia avisos verbosos da biblioteca

MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Carregando {MODELO} (uma vez para o notebook inteiro)...")
tok = AutoTokenizer.from_pretrained(MODELO)
llm = AutoModelForCausalLM.from_pretrained(
    MODELO, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None)
print("Pronto — GPU:", torch.cuda.is_available())

Carregando Qwen/Qwen2.5-1.5B-Instruct (uma vez para o notebook inteiro)...


Pronto — GPU: True


## Seção 1 — Agente ReAct mínimo *(slide 10)*

Um agente é um **loop**: o modelo pensa, escolhe uma ação em JSON, a aplicação
executa a ferramenta e devolve a observação — até o modelo finalizar.
Três peças: o prompt de formato, o loop `while` e o executor de ferramenta.
Sem framework.

### Primeiro, as ferramentas

Todas **somente leitura** — princípio do menor privilégio. Repare em duas
decisões de engenharia:

- `buscar_documento` reusa o corpus da Aula 02 e dá **mais peso a termos
  raros** — a mesma intuição do BM25 que estudamos lá;
- `calculadora` avalia a expressão pela árvore sintática (`ast`) com uma
  whitelist de operações. Por que não `eval()`? Porque `eval` executa
  **qualquer** código Python — jamais o use com texto vindo de um LLM.

In [2]:
import ast
import operator
import unicodedata

CLIMA = {
    "recife": "28°C, parcialmente nublado, 70% de umidade, vento 18 km/h",
    "sao paulo": "19°C, garoa, 85% de umidade, vento 10 km/h",
    "rio de janeiro": "26°C, ensolarado, 65% de umidade, vento 14 km/h",
    "brasilia": "24°C, céu limpo, 40% de umidade, vento 8 km/h",
    "porto alegre": "16°C, chuva fraca, 90% de umidade, vento 22 km/h",
}

PASTA_DOCS = Path("..") / ".." / "aula02-rag" / "handson" / "data"


def _normalizar(texto):
    sem_acento = unicodedata.normalize("NFD", texto)
    return "".join(c for c in sem_acento if not unicodedata.combining(c)).lower().strip()


def consultar_clima(cidade):
    """Clima atual da cidade (base estática local — sem rede, sem API)."""
    chave = _normalizar(cidade)
    if chave not in CLIMA:
        disponiveis = ", ".join(sorted(CLIMA))
        raise ValueError(f"cidade '{cidade}' não cadastrada. Disponíveis: {disponiveis}")
    return f"Clima em {cidade}: {CLIMA[chave]}"


def buscar_documento(consulta):
    """Busca lexical nos documentos internos, com peso maior para termos raros."""
    termos = {t for t in _normalizar(consulta).split() if len(t) > 3}
    paragrafos = []
    for arquivo in sorted(PASTA_DOCS.glob("*.md")):
        for paragrafo in arquivo.read_text(encoding="utf-8").split("\n\n"):
            paragrafos.append((arquivo.stem, paragrafo, _normalizar(paragrafo)))
    # termo raro vale mais: peso = 1 / (nº de parágrafos em que aparece)
    peso = {t: 1.0 / max(1, sum(t in norm for _, _, norm in paragrafos)) for t in termos}
    candidatos = []
    for origem, paragrafo, norm in paragrafos:
        pontos = sum(peso[t] for t in termos if t in norm)
        if pontos:
            candidatos.append((pontos, f"[{origem}] {' '.join(paragrafo.split())[:280]}"))
    if not candidatos:
        return f"Nenhum trecho encontrado para '{consulta}'. Tente outros termos."
    candidatos.sort(key=lambda par: -par[0])
    return "\n".join(trecho for _, trecho in candidatos[:2])


_OPERADORES = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
    ast.Mod: operator.mod,
}


def calculadora(expressao):
    """Aritmética com whitelist de operações via AST — nunca eval() com LLM."""
    def avaliar(no):
        if isinstance(no, ast.Expression):
            return avaliar(no.body)
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            return no.value
        if isinstance(no, ast.BinOp) and type(no.op) in _OPERADORES:
            return _OPERADORES[type(no.op)](avaliar(no.left), avaliar(no.right))
        if isinstance(no, ast.UnaryOp) and type(no.op) in _OPERADORES:
            return _OPERADORES[type(no.op)](avaliar(no.operand))
        raise ValueError(f"operação não permitida: {ast.dump(no)[:40]}")

    resultado = avaliar(ast.parse(expressao, mode="eval"))
    return f"{expressao} = {resultado}"


def consultar_cotacao(moeda):
    """Ferramenta propositalmente QUEBRADA — simula serviço externo fora do ar."""
    raise TimeoutError("serviço de cotações indisponível (timeout após 5 s)")


print("Ferramentas prontas:", ", ".join(["consultar_clima", "buscar_documento",
                                          "calculadora", "consultar_cotacao"]))

Ferramentas prontas: consultar_clima, buscar_documento, calculadora, consultar_cotacao


### Agora, o agente

In [3]:
FERRAMENTAS = {
    "consultar_clima": consultar_clima,
    "buscar_documento": buscar_documento,
    "calculadora": calculadora,
}

SISTEMA = """Você é um agente que resolve tarefas usando ferramentas.

FERRAMENTAS DISPONÍVEIS:
- consultar_clima(cidade): clima atual de uma cidade brasileira
- buscar_documento(consulta): busca trechos nos documentos internos da empresa
- calculadora(expressao): calcula uma expressão aritmética

FORMATO OBRIGATÓRIO — responda SEMPRE com exatamente estas duas linhas:
Pensamento: <raciocínio curto sobre o próximo passo>
Ação: {"ferramenta": "<nome>", "argumentos": {"<param>": "<valor>"}}

Quando já tiver tudo para responder, finalize com:
Pensamento: <raciocínio curto>
Ação: {"ferramenta": "finalizar", "argumentos": {"resposta": "<resposta completa ao usuário>"}}

Use UMA ferramenta por vez. Nunca invente o resultado de uma ferramenta.
A resposta final deve cobrir TODAS as partes da pergunta, usando as
observações recebidas, em no máximo 3 frases.

Exemplo:
Pergunta: Quanto é 15% de 340?
Pensamento: Preciso calcular 0.15 * 340.
Ação: {"ferramenta": "calculadora", "argumentos": {"expressao": "0.15 * 340"}}
Observação: 0.15 * 340 = 51.0
Pensamento: Já tenho o resultado.
Ação: {"ferramenta": "finalizar", "argumentos": {"resposta": "15% de 340 é 51."}}"""


def chamar_modelo(mensagens):
    entrada = tok.apply_chat_template(mensagens, add_generation_prompt=True,
                                      return_tensors="pt").to(llm.device)
    with torch.no_grad():
        saida = llm.generate(entrada, max_new_tokens=300, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True).strip()


def extrair_acao(texto):
    """Parser tolerante: pega o primeiro objeto JSON depois de 'Ação:'."""
    inicio = texto.find("{", texto.find("Ação:"))
    profundidade = 0
    for i in range(inicio, len(texto)):
        profundidade += (texto[i] == "{") - (texto[i] == "}")
        if profundidade == 0:
            return json.loads(texto[inicio:i + 1])
    raise ValueError("nenhuma Ação em JSON encontrada na resposta do modelo")

In [4]:
def executar_agente(pergunta, max_iteracoes=6):
    print("=" * 70)
    print(f"PERGUNTA: {pergunta}")
    mensagens = [{"role": "system", "content": SISTEMA},
                 {"role": "user", "content": f"Pergunta: {pergunta}"}]

    for iteracao in range(1, max_iteracoes + 1):
        saida = chamar_modelo(mensagens)
        saida = saida.split("Observação:")[0].strip()   # o modelo não inventa observação
        print("-" * 70)
        print(f"[ciclo {iteracao}]")
        print(saida)
        acao = extrair_acao(saida)

        if acao["ferramenta"] == "finalizar":
            print("=" * 70)
            print(f"RESPOSTA FINAL: {acao['argumentos']['resposta']}")
            return

        funcao = FERRAMENTAS[acao["ferramenta"]]
        try:
            observacao = funcao(**acao["argumentos"])
        except Exception as erro:            # erro vira observação
            observacao = f"ERRO da ferramenta: {erro}"
        print(f"Observação: {observacao}")
        mensagens.append({"role": "assistant", "content": saida})
        mensagens.append({"role": "user", "content": f"Observação: {observacao}"})

    print("Limite de iterações atingido sem resposta final.")


executar_agente("Qual o clima em Recife agora e o que a política de viagens "
                "diz sobre deslocamento local?")

PERGUNTA: Qual o clima em Recife agora e o que a política de viagens diz sobre deslocamento local?


----------------------------------------------------------------------
[ciclo 1]
Pensamento: Primeiro preciso obter o clima em Recife. Depois, vou pesquisar a política de viagens para descobrir se há restrições ou recomendações sobre deslocamentos locais.

Ação: {"ferramenta": "consultar_clima", "argumentos": {"cidade": "Recife"}}
Observação: Clima em Recife: 28°C, parcialmente nublado, 70% de umidade, vento 18 km/h


----------------------------------------------------------------------
[ciclo 2]
Pensamento: Agora que tenho o clima em Recife, posso verificar a política de viagens. Ainda não encontrei informações específicas sobre viagens locais, mas posso perguntar se há restrições ou recomendações.

Ação: {"ferramenta": "buscar_documento", "argumentos": {"consulta": "Política de viagens - Deslocamento local"}}
Observação: [politica-viagens] A hospedagem deve respeitar o teto diário definido por cidade na tabela do portal interno. O deslocamento local deve priorizar aplicativos de transporte corporativo cadastrados; táxis são aceitos quando não houver alternativa. Aluguel de veículo exige justificativa e aprovação do
[politica-viagens] # Política de Viagens Corporativas — Innovatech Soluções


----------------------------------------------------------------------
[ciclo 3]
Pensamento: Com base nas informações obtidas, posso formular uma resposta sobre o clima em Recife e a política de viagens.

Ação: {"ferramenta": "finalizar", "argumentos": {"resposta": "Atualmente, o clima em Recife é 28°C, parcialmente nublado, 70% de umidade e com vento moderado. A política de viagens sugere que a hospedagem respeite o teto diário definido pela cidade, preferindo os aplicativos de transporte corporativo cadastrados. Táxis são aceitos apenas quando não existir outra opção disponível. Para deslocamentos locais, o aluguel de veículo requer justificativa e aprovação."}}
RESPOSTA FINAL: Atualmente, o clima em Recife é 28°C, parcialmente nublado, 70% de umidade e com vento moderado. A política de viagens sugere que a hospedagem respeite o teto diário definido pela cidade, preferindo os aplicativos de transporte corporativo cadastrados. Táxis são aceitos apenas quando não existir outra opção di

## Seção 2 — Contratos: schema, validação, limites e auditoria *(slides 13–14 e 19)*

O que esta seção adiciona ao agente mínimo:
1. **Schemas pydantic** por ferramenta — argumento inválido NÃO executa.
2. **Erro vira OBSERVAÇÃO** — o agente lê e se corrige.
3. **Limite de iterações** — loop nunca roda para sempre.
4. **Log de auditoria** em `logs/audit_log.jsonl` — cada decisão rastreável.

Primeiro, o contrato em ação com chamadas FORJADAS (sem modelo) — inclusive
uma tentativa de injetar `__import__('os')` na calculadora.

In [5]:
class ArgsClima(BaseModel):
    cidade: str = Field(min_length=2, description="nome da cidade brasileira")

class ArgsBusca(BaseModel):
    consulta: str = Field(min_length=4, description="termos a buscar nos documentos")

class ArgsCalculadora(BaseModel):
    expressao: str = Field(pattern=r"^[\d\s+\-*/().,%]+$",
                           description="apenas números e operadores aritméticos")

class ArgsCotacao(BaseModel):
    moeda: str = Field(min_length=3, max_length=3, description="código ISO, ex.: USD")

class ChamadaFerramenta(BaseModel):
    ferramenta: Literal["consultar_clima", "buscar_documento", "calculadora",
                        "consultar_cotacao", "finalizar"]
    argumentos: dict

REGISTRO = {
    "consultar_clima": (consultar_clima, ArgsClima),
    "buscar_documento": (buscar_documento, ArgsBusca),
    "calculadora": (calculadora, ArgsCalculadora),
    "consultar_cotacao": (consultar_cotacao, ArgsCotacao),
}


def validar_e_executar(acao_bruta):
    """Valida contra o contrato e só então executa. Retorna (observação, erro)."""
    try:
        chamada = ChamadaFerramenta.model_validate(acao_bruta)
    except ValidationError:
        return (f"ERRO de contrato: ferramenta desconhecida. Válidas: "
                f"{', '.join(REGISTRO)} ou finalizar", "contrato")
    funcao, schema_args = REGISTRO[chamada.ferramenta]
    try:
        argumentos = schema_args.model_validate(chamada.argumentos)
    except ValidationError as e:
        detalhe = e.errors()[0]
        return (f"ERRO de validação em '{detalhe['loc'][0]}': {detalhe['msg']}. "
                f"Corrija os argumentos e tente de novo.", "validacao")
    try:
        return funcao(**argumentos.model_dump()), None
    except Exception as erro:                # erro de execução vira observação
        return f"ERRO da ferramenta: {erro}", "execucao"


print("O CONTRATO EM AÇÃO — chamadas forjadas, sem modelo:")
FORJADAS = [
    {"ferramenta": "apagar_banco", "argumentos": {}},
    {"ferramenta": "calculadora", "argumentos": {"expressao": "__import__('os')"}},
    {"ferramenta": "calculadora", "argumentos": {"expressao": "340 * 0.15"}},
]
for acao in FORJADAS:
    observacao, erro = validar_e_executar(acao)
    situacao = "BLOQUEADA" if erro in ("contrato", "validacao") else "executada"
    print(f"  {json.dumps(acao, ensure_ascii=False)[:64]:64s} -> [{situacao}]")
    print(f"      {observacao}")

O CONTRATO EM AÇÃO — chamadas forjadas, sem modelo:
  {"ferramenta": "apagar_banco", "argumentos": {}}                 -> [BLOQUEADA]
      ERRO de contrato: ferramenta desconhecida. Válidas: consultar_clima, buscar_documento, calculadora, consultar_cotacao ou finalizar
  {"ferramenta": "calculadora", "argumentos": {"expressao": "__imp -> [BLOQUEADA]
      ERRO de validação em 'expressao': String should match pattern '^[\d\s+\-*/().,%]+$'. Corrija os argumentos e tente de novo.
  {"ferramenta": "calculadora", "argumentos": {"expressao": "340 * -> [executada]
      340 * 0.15 = 51.0


### Trace de execução: reprodutibilidade para agentes

Agentes são difíceis de depurar justamente porque cada execução pode tomar um
caminho diferente. A solução de engenharia é **gravar o trace**: cada decisão
do modelo vai para `logs/trace_agente.json`, e uma execução salva pode ser
**reproduzida** depois, passo a passo, sem chamar o modelo — para depurar um
incidente, escrever um teste de regressão ou demonstrar um comportamento.

- `GRAVAR_TRACE=True` → salva as decisões desta execução;
- `REPRODUZIR_TRACE=True` → reexecuta as decisões salvas (instantâneo e determinístico).

In [6]:
GRAVAR_TRACE = True        # salva as decisões do modelo nesta execução
REPRODUZIR_TRACE = False   # True = reexecuta o trace salvo, sem chamar o modelo

MAX_ITERACOES = 4
PASTA_LOGS = Path("logs")
PASTA_LOGS.mkdir(exist_ok=True)
ARQ_AUDITORIA = PASTA_LOGS / "audit_log.jsonl"
ARQ_TRACE = PASTA_LOGS / "trace_agente.json"

trace = json.loads(ARQ_TRACE.read_text(encoding="utf-8")) if REPRODUZIR_TRACE else {}
print(f"GRAVAR_TRACE={GRAVAR_TRACE}  REPRODUZIR_TRACE={REPRODUZIR_TRACE}")

GRAVAR_TRACE=True  REPRODUZIR_TRACE=False


In [7]:
SISTEMA_V2 = """Você é um agente que resolve tarefas usando ferramentas.

FERRAMENTAS DISPONÍVEIS:
- consultar_clima(cidade): clima atual de uma cidade brasileira
- buscar_documento(consulta): busca trechos nos documentos internos da empresa
- calculadora(expressao): calcula uma expressão aritmética
- consultar_cotacao(moeda): cotação de uma moeda (código ISO de 3 letras, ex.: USD)

FORMATO OBRIGATÓRIO — responda SEMPRE com exatamente estas duas linhas:
Pensamento: <raciocínio curto sobre o próximo passo>
Ação: {"ferramenta": "<nome>", "argumentos": {"<param>": "<valor>"}}

Quando já tiver tudo para responder (ou concluir que não é possível), finalize:
Pensamento: <raciocínio curto>
Ação: {"ferramenta": "finalizar", "argumentos": {"resposta": "<resposta ao usuário>"}}

Use UMA ferramenta por vez. Nunca invente o resultado de uma ferramenta.
Se uma ferramenta falhar duas vezes, finalize explicando a limitação."""


def auditar(**evento):
    evento["timestamp"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    with open(ARQ_AUDITORIA, "a", encoding="utf-8") as f:
        f.write(json.dumps(evento, ensure_ascii=False) + "\n")


def executar_agente_v2(rotulo, pergunta):
    print("=" * 70)
    print(f"CENÁRIO: {rotulo}")
    print(f"PERGUNTA: {pergunta}")
    mensagens = [{"role": "system", "content": SISTEMA_V2},
                 {"role": "user", "content": f"Pergunta: {pergunta}"}]
    saidas_gravadas = trace.setdefault(rotulo, [])

    for iteracao in range(1, MAX_ITERACOES + 1):
        if REPRODUZIR_TRACE:
            saida = saidas_gravadas[iteracao - 1]      # decisões gravadas no ensaio
        else:
            saida = chamar_modelo(mensagens).split("Observação:")[0].strip()
            if GRAVAR_TRACE:
                saidas_gravadas.append(saida)
        print("-" * 70)
        print(f"[ciclo {iteracao}/{MAX_ITERACOES}]")
        print(saida)

        try:
            acao = extrair_acao(saida)
        except (ValueError, json.JSONDecodeError) as e:
            observacao, erro = f"ERRO de formato: {e}. Use o formato exigido.", "formato"
            acao = {"ferramenta": "?", "argumentos": {}}
        else:
            if acao.get("ferramenta") == "finalizar":
                auditar(cenario=rotulo, iteracao=iteracao, ferramenta="finalizar",
                        argumentos=acao.get("argumentos", {}), erro=None)
                print("=" * 70)
                print(f"RESPOSTA FINAL: {acao['argumentos'].get('resposta', '')}")
                return
            observacao, erro = validar_e_executar(acao)

        print(f"Observação: {observacao}")
        auditar(cenario=rotulo, iteracao=iteracao, ferramenta=acao.get("ferramenta"),
                argumentos=acao.get("argumentos"), resultado=str(observacao)[:120],
                erro=erro)
        mensagens.append({"role": "assistant", "content": saida})
        mensagens.append({"role": "user", "content": f"Observação: {observacao}"})

    print("=" * 70)
    print(f"LIMITE DE {MAX_ITERACOES} ITERAÇÕES ATINGIDO — o loop foi interrompido "
          "com segurança (sem resposta, mas sem custo infinito).")
    auditar(cenario=rotulo, iteracao=MAX_ITERACOES, ferramenta=None,
            argumentos=None, erro="limite_iteracoes")

In [8]:
executar_agente_v2("erro vira observação e o agente se corrige",
                   "Qual o clima em Fernando de Noronha?")

CENÁRIO: erro vira observação e o agente se corrige
PERGUNTA: Qual o clima em Fernando de Noronha?


----------------------------------------------------------------------
[ciclo 1/4]
Pensamento: Preciso consultar o clima em Fernando de Noronha.
Ação: {"ferramenta": "consultar_clima", "argumentos": {"cidade": "Fernando de Noronha"}}
Observação: ERRO da ferramenta: cidade 'Fernando de Noronha' não cadastrada. Disponíveis: brasilia, porto alegre, recife, rio de janeiro, sao paulo


----------------------------------------------------------------------
[ciclo 2/4]
Pensamento: A cidade informada não está disponível na lista dos dados disponíveis.
Ação: {"ferramenta": "finalizar", "argumentos": {"resposta": "Desculpe, mas parece haver um erro no nome da cidade. As opções disponíveis são: brasília, porto alegre, recife, rio de janeiro e sãopaulo."}}
RESPOSTA FINAL: Desculpe, mas parece haver um erro no nome da cidade. As opções disponíveis são: brasília, porto alegre, recife, rio de janeiro e sãopaulo.


In [9]:
executar_agente_v2("ferramenta quebrada + limite de iterações",
                   "Qual a cotação do dólar (USD) agora?")

CENÁRIO: ferramenta quebrada + limite de iterações
PERGUNTA: Qual a cotação do dólar (USD) agora?


----------------------------------------------------------------------
[ciclo 1/4]
Pensamento: Preciso consultar a cotação do dólar (USD).
Ação: {"ferramenta": "consultar_cotacao", "argumentos": {"moeda": "USD"}}
Observação: ERRO da ferramenta: serviço de cotações indisponível (timeout após 5 s)


----------------------------------------------------------------------
[ciclo 2/4]
Pensamento: A ferramenta retornou um erro e não conseguiu obter a cotação do dólar (USD). Infelizmente, não consigo fornecer essa informação neste momento.
Observação: ERRO de formato: Expecting value: line 1 column 1 (char 0). Use o formato exigido.


----------------------------------------------------------------------
[ciclo 3/4]
Desculpe pela confusão anterior. Vamos tentar novamente:

Pensamento: Preciso consultar a cotação do dólar (USD).
Ação: {"ferramenta": "consultar_cotacao", "argumentos": {"moeda": "USD"}}
Observação: ERRO da ferramenta: serviço de cotações indisponível (timeout após 5 s)


----------------------------------------------------------------------
[ciclo 4/4]
Desculpe pela repetição anterior. Infelizmente, parece haver um problema com a disponibilidade dos serviços de cotação naquele momento. Não posso fornecer informações financeiras diretamente.
Observação: ERRO de formato: Expecting value: line 1 column 1 (char 0). Use o formato exigido.
LIMITE DE 4 ITERAÇÕES ATINGIDO — o loop foi interrompido com segurança (sem resposta, mas sem custo infinito).


In [10]:
if GRAVAR_TRACE and not REPRODUZIR_TRACE:
    ARQ_TRACE.write_text(json.dumps(trace, ensure_ascii=False, indent=1),
                         encoding="utf-8")
    print(f"Trace gravado em {ARQ_TRACE} — experimente reexecutar esta seção")
    print("com REPRODUZIR_TRACE=True e compare: mesmas decisões, sem modelo.")

print(f"\nAUDITORIA — últimas linhas de {ARQ_AUDITORIA.name}:")
for linha in ARQ_AUDITORIA.read_text(encoding="utf-8").strip().splitlines()[-3:]:
    registro = json.loads(linha)
    print(f"  it{registro['iteracao']} cenário='{registro['cenario'][:30]}' "
          f"ferramenta={registro['ferramenta']} erro={registro['erro']}")
print("\nCada linha tem timestamp, ferramenta, argumentos e erro — o caminho")
print("de decisão inteiro, auditável (ponte para a observabilidade e a Aula 06).")

Trace gravado em logs\trace_agente.json — experimente reexecutar esta seção
com REPRODUZIR_TRACE=True e compare: mesmas decisões, sem modelo.

AUDITORIA — últimas linhas de audit_log.jsonl:
  it3 cenário='ferramenta quebrada + limite d' ferramenta=consultar_cotacao erro=execucao
  it4 cenário='ferramenta quebrada + limite d' ferramenta=? erro=formato
  it4 cenário='ferramenta quebrada + limite d' ferramenta=None erro=limite_iteracoes

Cada linha tem timestamp, ferramenta, argumentos e erro — o caminho
de decisão inteiro, auditável (ponte para a observabilidade e a Aula 06).
